# Try FR3 Cartesian target control online

Start with a stationary robot, make one small move, then change one setting.
Each example builds on the previous one. **Select a code cell and press Shift+Enter**
to run it and move to the next cell. You can also choose **Run → Run All Cells**.

Your Python commands use the native Rust controller and a real `franka-sim` process
inside this Binder session. Each example starts a fresh simulated FR3 and closes it
before returning, so you can rerun examples without keeping a robot connection open.
Startup can take a few seconds on a shared host.

This is a temporary playground: download the notebook to keep your edits before
closing the session. Nothing here connects to a physical robot.

## 1. Load the lab

Run this once. The small lab helpers handle simulator startup, recording, and cleanup;
you will see the underlying `franka` control code further down.

In [ ]:
from sim_lab import preview_robot, single_move, replay, compare_runs

print("Ready. Next, display the stationary robot.")

## 2. Meet the robot

Run this cell first and check that you can see the robot below it. The preview uses
the simulated joint positions. It does not send a movement command.

In [ ]:
preview_robot()

## 3. Make one small move

Move the target **2 cm along x**. All distances are in metres, so `0.02` means 2 cm.
The helper sends `arm.move_by([0.02, 0.0, 0.0])` and records the response for two seconds.
When the recording finishes, the next cell plays back the measured joint motion.
This browser animation shows the actual simulation trace, not a second physics model.

In [ ]:
first_move = single_move([0.02, 0.0, 0.0])

In [ ]:
replay(first_move)

## 4. Change just the speed limit

Repeat the same move from the same starting pose, with a lower velocity limit.
`0.01` means **1 cm per second**. The acceleration and jerk limits stay at their defaults.
The lower limit should make this short movement easier to follow. Run the recording
cell, then the playback cell.

In [ ]:
slower_move = single_move([0.02, 0.0, 0.0], max_velocity=0.01)

In [ ]:
replay(slower_move)

Compare the measured x displacement for those two runs. A velocity limit is an
upper bound: the arm also needs time to accelerate, and a short move may never
reach the limit. The slower run may still be approaching its destination when the
two-second recording ends.

In [ ]:
compare_runs([first_move, slower_move]);

## 5. Try acceleration and jerk

Now edit one number in the following cell and rerun it. Keep the displacement the
same so you can compare how the motion changes.

| Setting | What it limits | Unit |
| --- | --- | --- |
| `max_velocity` | Translation speed | m/s |
| `max_acceleration` | How fast velocity changes | m/s² |
| `max_jerk` | How fast acceleration changes | m/s³ |

Try lowering `max_acceleration` to `0.05`, or `max_jerk` to `1.0`. Change just one
setting at a time. Increase `hold_seconds` if you want to record a slow move longer.

In [ ]:
edited_move = single_move(
    [0.02, 0.0, 0.0],
    max_velocity=0.3,
    max_acceleration=0.5,
    max_jerk=20.0,
    hold_seconds=2.0,
)

In [ ]:
replay(edited_move)

## 6. Use the control API directly

The short examples above wrap the setup and recording. This next example exposes
those pieces so you can write your own movement sequence.

`arm.move_by([dx, dy, dz])` offsets the **current target**, not the measured position.
`arm.move_to([x, y, z])` sets an absolute target. Both use metres.

First load the recording tools and choose parameters for this longer experiment.

In [ ]:
import time
import numpy as np
import franka
from sim_runtime import simulator
from sim_lab import (
    Parameters, measured_position,
    plot_run, run_experiment,
)

if "runs" not in globals():
    runs = []

In [ ]:
# EDIT THESE VALUES, then rerun this cell and the motion cell.
max_velocity = 0.3       # m/s
max_acceleration = 0.5   # m/s²
max_jerk = 20.0          # m/s³
travel = 0.04           # 4 cm along x
hold_seconds = 1.5      # time between target updates

parameters = Parameters(
    max_velocity=max_velocity,
    max_acceleration=max_acceleration,
    max_jerk=max_jerk,
    amplitude=travel,
    hold_seconds=hold_seconds,
)
parameters.validate()


### Move forward, backward, then return

The movement commands are marked **YOUR MOVEMENT CODE** below. The `observe(...)`
function only samples states; the native controller keeps
running between Python calls. Try a different offset or add another `arm.move_to(...)`
followed by `observe(...)`.

The context managers close the controller and simulator before the cell finishes.
The visualization shows the measured arm and requested destination; it is not a
hardware-fidelity model of a physical FR3.

In [ ]:
if globals().get("active_task") is not None and not active_task.done():
    raise RuntimeError("Stop the slider experiment and wait before running this cell")

rows = []

with simulator() as address:
    robot = arm = None
    try:
        robot = franka.Robot(address, realtime="ignore")
        model = robot.model()
        robot.set_collision_behavior_simple([40.] * 7, [40.] * 7, [40.] * 6, [40.] * 6)

        with robot.cartesian_targets(
            max_velocity=max_velocity,
            max_acceleration=max_acceleration,
            max_jerk=max_jerk,
            backend="impedance",
        ) as arm:
            start = np.array(arm.target()[:3], copy=True)
            wall_start = time.monotonic()

            def observe(seconds):
                deadline = time.monotonic() + seconds
                while time.monotonic() < deadline and arm.running:
                    state = arm.state()
                    if not rows or float(state.time) > rows[-1][0]:
                        q = np.array(state.q, copy=True)
                        target = np.array(arm.target()[:3], copy=True)
                        measured = measured_position(model, state, "impedance")
                        wire = np.array(state.O_T_EE[:3, 3], copy=True)
                        elapsed = time.monotonic() - wall_start
                        rows.append((float(state.time), elapsed, q, target, measured, wire))
                    time.sleep(0.02)

            # YOUR MOVEMENT CODE: try changing these targets.
            arm.move_by([travel, 0.0, 0.0])
            observe(hold_seconds)
            arm.move_by([-2 * travel, 0.0, 0.0])
            observe(hold_seconds)
            arm.move_to(start)
            observe(hold_seconds)
    finally:
        # Drop native handles before stopping the simulator, including on errors.
        arm = None
        robot = None

# All native motion and simulator processes have stopped here.
if len(rows) < 4:
    raise RuntimeError("Too few samples; increase hold_seconds and run again.")
stamps, wall, q, target, measured, wire = map(np.asarray, zip(*rows))
result = dict(
    parameters=parameters, time=stamps - stamps[0], wall=wall,
    q=q, target=target, measured=measured, wire_measured=wire,
    measured_frame="FK EE from q and tool transforms", model=model,
)
runs.append(result)
print(f"Recorded {len(rows)} states. Controller closed; simulator stopped.")


### Watch the recorded motion

Play, pause, or scrub the measured joint trajectory below. The orange marker is the
requested destination; the green marker is the measured end-effector position.

In [ ]:
replay(result)

### Read the full response

The dashed line is the requested destination, **not the smoothed controller
reference**. The measured position follows it with a delay. The impedance controller
also includes compliance and a following-distance leash, so doubling a limit does
not necessarily double the measured peak.

The derivative plots are approximate finite differences of sampled motion. They
help compare runs but do **not** certify the controller's exact 1 kHz velocity,
acceleration, or jerk limits. Jerk is especially sensitive to sampling noise.

**Coordinate frames:** this simulator's wire `O_T_EE` reports joint-7 rather than the
controller's end-effector frame. We use `model.pose("ee", state.q, state.F_T_EE,
state.EE_T_K)` for the impedance controller comparison and preserve the unmodified
wire positions in `result["wire_measured"]`.


In [ ]:
plot_run(result)
if len(runs) > 1:
    compare_runs(runs)


### Extend your experiment

1. Change only `max_jerk` to `1.0`; rerun the parameter, motion, and plot cells.
2. Restore jerk and lower `max_velocity` to `0.05`. Does the arm finish each segment?
3. Replace a movement with `arm.move_to(start + [0.02, 0.01, 0.0])`.
4. Increase `hold_seconds` to give a slow trajectory more time to settle.

Each motion cell starts from a new simulation. `runs` retains the measured traces
inside this kernel; the comparison aligns each run's first measured position.
Keep the same target sequence and timing when comparing limits.

To save one trace, uncomment the following cell. Download the CSV through Jupyter's
file browser before the session ends.


In [ ]:
# np.savetxt(
#     "my-fr3-run.csv",
#     np.column_stack([result["time"], result["target"], result["measured"]]),
#     delimiter=",", header="time,target_x,target_y,target_z,measured_x,measured_y,measured_z",
#     comments="",
# )


## Optional: explore with sliders

These controls run the helper's fixed sequence: hold the start, move forward, move
backward, return to the start. Sliders affect the **next** run. Nothing starts merely
by executing this cell or using Run All. Click **Run experiment** to begin; **Stop**
requests orderly shutdown, which may take time to settle. Wait for completion before
running the teaching cells above again. Every slider experiment also gets a fresh simulator.
Its recorded motion appears as browser playback after the run finishes.


In [ ]:
import asyncio
import threading
import ipywidgets as widgets
from IPython.display import display, clear_output

if "active_task" not in globals():
    active_task = None

if active_task is not None and not active_task.done():
    raise RuntimeError("Stop the existing run and wait before recreating the controls")

# Retire the previous panel before rebinding callback globals on cell rerun.
for old_button, old_callback in globals().get("_panel_callbacks", []):
    old_button.on_click(old_callback, remove=True)
    old_button.disabled = True
for old_widget in globals().get("_panel_widgets", []):
    old_widget.close()
_panel_callbacks = []
_panel_widgets = []

velocity = widgets.FloatSlider(value=.3, min=.01, max=.8, step=.01, description="v (m/s)")
acceleration = widgets.FloatSlider(value=.5, min=.05, max=3., step=.05, description="a (m/s²)")
jerk = widgets.FloatSlider(value=20., min=.1, max=80., step=.1, description="j (m/s³)")
amplitude = widgets.FloatSlider(value=.04, min=.005, max=.08, step=.005, description="travel (m)")
hold = widgets.FloatSlider(value=1.5, min=.5, max=5., step=.5, description="hold (s)")
backend = widgets.Dropdown(options=["impedance", "robot"], value="impedance", description="backend")
run_button = widgets.Button(description="Run experiment", button_style="success")
stop_button = widgets.Button(description="Stop", button_style="warning", disabled=True)
status = widgets.HTML(value="Ready — no robot connection yet")
viewer_output, plot_output = widgets.Output(), widgets.Output()
controls = [velocity, acceleration, jerk, amplitude, hold, backend]
stop_event = threading.Event()

async def experiment(parameters):
    global active_task
    try:
        with viewer_output:
            clear_output(wait=True)
            print("Recording simulation; playback appears when the run finishes.")
        worker = asyncio.create_task(asyncio.to_thread(run_experiment, parameters, stop_event, False))
        try:
            result = await asyncio.shield(worker)
        except asyncio.CancelledError:
            # Cancelling a Python task cannot cancel the native worker thread.
            # Request a stop and join before making the controls available again.
            stop_event.set()
            status.value = "Task cancelled; waiting for the controller to stop…"
            try:
                await asyncio.shield(worker)
            finally:
                status.value = "Cancelled; controller worker has finished"
            raise
        runs.append(result)
        with viewer_output:
            clear_output(wait=True)
            display(replay(result))
        missed = sum(not segment["reached"] for segment in result["segments"])
        with plot_output:
            clear_output(wait=True)
            plot_run(result)
        status.value = f"Run {len(runs)} {'stopped' if result['stopped'] else 'complete'}; controller closed; {missed} segments ended >5 mm from target"
    except Exception as error:
        # Do not retain exception objects/native robot references in notebook history.
        status.value = "Run failed; inspect the printed simulator error"
        with plot_output:
            print(type(error).__name__ + ": " + str(error))
    finally:
        for control in controls:
            control.disabled = False
        run_button.disabled, stop_button.disabled = False, True


def start(_):
    global active_task
    if active_task is not None and not active_task.done():
        return
    parameters = Parameters(velocity.value, acceleration.value, jerk.value,
                            amplitude.value, hold.value, backend=backend.value)
    parameters.validate()
    stop_event.clear()
    for control in controls:
        control.disabled = True
    run_button.disabled, stop_button.disabled = True, False
    status.value = "Running on the local simulator…"
    active_task = asyncio.create_task(experiment(parameters))


def stop(_):
    stop_event.set()
    status.value = "Stopping; waiting for the Rust controller to settle and close…"

run_button.on_click(start)
stop_button.on_click(stop)
display(widgets.VBox(controls + [widgets.HBox([run_button, stop_button]), status]),
        viewer_output, plot_output)

# Keep exact callback identities so an old panel cannot start a new run.
_panel_callbacks = [(run_button, start), (stop_button, stop)]
_panel_widgets = controls + [run_button, stop_button, status, viewer_output, plot_output]
